In [ ]:
# ============================================================
# Cell 1: Top-level imports + module reload
# ============================================================

import re
import gc
import json
import time
import random
import hashlib
import importlib
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

try:
    import torch
except ImportError:
    torch = None

import config
import metrics
import model_loader
import modes
import workloads
import runner
import benchmark_modes
import reporter

for module in [
    config,
    metrics,
    model_loader,
    modes,
    workloads,
    runner,
    benchmark_modes,
    reporter,
]:
    importlib.reload(module)

from config import CONFIG, RAW_RESULTS_DIR, BENCHMARK_DATA_DIR
from modes import build_runtime_mode_by_name
from workloads import (
    build_runtime_workload_by_name,
    build_runtime_workloads_for_name,
)
from runner import run_single_benchmark
from benchmark_modes import (
    save_results_json,
    save_results_csv,
    save_summary_csv,
    build_aggregate_rows,
    build_comparison_rows,
    save_aggregate_csv,
    apply_external_score_sidecar,
    annotate_results_with_baseline_similarity,
)
from reporter import generate_full_report

print("Project root:", Path.cwd())
print("RAW_RESULTS_DIR:", RAW_RESULTS_DIR)
print("BENCHMARK_DATA_DIR:", BENCHMARK_DATA_DIR)

In [ ]:
# ============================================================
# Cell 2: Notebook helpers
# ============================================================

def resolve_runtime_mode(mode_name: str):
    return build_runtime_mode_by_name(mode_name)


def resolve_runtime_workload(workload_name: str):
    match = re.fullmatch(r"(shared_prefix_chat)_v(\d+)", workload_name)
    if match:
        base_name = match.group(1)
        variant_id = int(match.group(2))
        return build_runtime_workload_by_name(
            base_name,
            repeated_prefix_variant=variant_id,
        )

    return build_runtime_workload_by_name(workload_name)


def expand_workload_group_names(workload_group_names):
    expanded = []

    for workload_name in workload_group_names:
        if re.fullmatch(r"shared_prefix_chat_v\d+", workload_name):
            expanded.append(workload_name)
            continue

        runtime_rows = build_runtime_workloads_for_name(workload_name)
        expanded.extend([row.name for row in runtime_rows])

    return list(dict.fromkeys(expanded))


def canonical_workload_group_name(expanded_workload_name: str) -> str:
    if "__" in str(expanded_workload_name):
        return str(expanded_workload_name).split("__", 1)[0]
    return str(expanded_workload_name)


def run_one(mode_name: str, workload_name: str, trial_index: int = 0):
    runtime_mode = resolve_runtime_mode(mode_name)
    runtime_workload = resolve_runtime_workload(workload_name)

    return run_single_benchmark(
        runtime_mode=runtime_mode,
        workload=runtime_workload,
        trial_index=trial_index,
    )


def fmt_num(x, decimals=2):
    if x is None or pd.isna(x):
        return "NA"
    return f"{x:.{decimals}f}"


def count_jsonl_rows(path: Path) -> int:
    if not path.exists():
        return 0

    n = 0
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                n += 1

    return n


def preview_jsonl(path: Path, n=2):
    print("\nPreview:", path)

    if not path.exists():
        print("Missing file.")
        return

    with open(path, "r", encoding="utf-8") as f:
        for idx, line in enumerate(f):
            if idx >= n:
                break
            print(line.strip())

In [ ]:
# ============================================================
# Cell 3: Verify benchmark sidecars
# ============================================================

SIDECARE_FILES = {
    "mmlu_pro_eval": "mmlu_pro_eval.jsonl",
    "gsm8k_eval": "gsm8k_eval.jsonl",
    "truthfulqa_eval": "truthfulqa_eval.jsonl",
    "gpqa_eval": "gpqa_eval.jsonl",
    "mlu_eval": "mlu_eval.jsonl",
    "mt_bench_eval": "mt_bench_eval.jsonl",
    "alpacaeval2_lc_eval": "alpacaeval2_lc_eval.jsonl",
}

sidecar_rows = []

for group_name, filename in SIDECARE_FILES.items():
    path = BENCHMARK_DATA_DIR / filename
    sidecar_rows.append({
        "workload_group": group_name,
        "path": str(path),
        "exists": path.exists(),
        "rows": count_jsonl_rows(path),
    })

sidecar_df = pd.DataFrame(sidecar_rows)
display(sidecar_df)

missing = sidecar_df[~sidecar_df["exists"]]
if len(missing) > 0:
    raise FileNotFoundError("Missing sidecars:\n" + "\n".join(missing["path"].tolist()))

for group_name, filename in SIDECARE_FILES.items():
    preview_jsonl(BENCHMARK_DATA_DIR / filename, n=1)

In [ ]:
# ============================================================
# Cell 4: Final sweep configuration
# ============================================================

STANDARD_DENSE_MODES = [
    "fp16_baseline",
    "gptq_4bit",
    "speculative_decoding",
    "gptq_plus_prefix_caching",
    "int8_plus_continuous_batching",
    "prefix_caching",
    "continuous_batching",
    "controller_v1",
]

BENCHMARK_DENSE_MODES = [
    "fp16_baseline",
    "gptq_4bit",
    "int8_quant",
    "speculative_decoding",
    "prefix_caching",
    "controller_v1",
]

NORMAL_WORKLOAD_GROUPS = {
    "short_prompt_short_output",
    "short_prompt_long_output",
    "long_prompt_short_output",
    "long_prompt_long_output",
    "shared_prefix_chat_v0",
    "shared_prefix_chat_v1",
    "memory_pressure_long_context",
}

AUTOMATIC_ACCURACY_WORKLOAD_GROUPS = {
    "mmlu_pro_eval",
    "gsm8k_eval",
    "truthfulqa_eval",
    "gpqa_eval",
    "mlu_eval",
}

EXTERNAL_JUDGE_WORKLOAD_GROUPS = {
    "mt_bench_eval",
    "alpacaeval2_lc_eval",
}

BENCHMARK_WORKLOAD_GROUPS = (
    AUTOMATIC_ACCURACY_WORKLOAD_GROUPS
    | EXTERNAL_JUDGE_WORKLOAD_GROUPS
)

FINAL_DENSE_WORKLOAD_GROUPS = [
    "short_prompt_short_output",
    "short_prompt_long_output",
    "long_prompt_short_output",
    "long_prompt_long_output",
    "shared_prefix_chat",
    "memory_pressure_long_context",
    "mmlu_pro_eval",
    "gsm8k_eval",
    "truthfulqa_eval",
    "gpqa_eval",
    "mlu_eval",
    "mt_bench_eval",
    "alpacaeval2_lc_eval",
]

# Normal fixed workloads get repeated trials.
NORMAL_NUM_TRIALS = 5 # Change to 15 or 20

# Benchmark-style workloads use many unique prompts and only one generation per prompt/mode.
BENCHMARK_NUM_TRIALS = 1

# Random prompt caps per benchmark group.
# MT-Bench only has 80, so use all 80.
WORKLOAD_SAMPLE_LIMITS = {
    "mmlu_pro_eval": 500,
    "gsm8k_eval": 500,
    "truthfulqa_eval": 500,
    "gpqa_eval": 500,
    "mlu_eval": 500,
    "alpacaeval2_lc_eval": 500,
    "mt_bench_eval": 80,
}

WORKLOAD_SAMPLE_SEED = 42

STANDARD_DENSE_MODES = list(dict.fromkeys(STANDARD_DENSE_MODES))
BENCHMARK_DENSE_MODES = list(dict.fromkeys(BENCHMARK_DENSE_MODES))
FINAL_DENSE_WORKLOAD_GROUPS = list(dict.fromkeys(FINAL_DENSE_WORKLOAD_GROUPS))

def trials_for_workload(expanded_workload_name: str) -> int:
    group_name = canonical_workload_group_name(expanded_workload_name)

    if group_name in BENCHMARK_WORKLOAD_GROUPS:
        return BENCHMARK_NUM_TRIALS

    return NORMAL_NUM_TRIALS


def modes_for_workload(expanded_workload_name: str):
    group_name = canonical_workload_group_name(expanded_workload_name)

    if group_name in BENCHMARK_WORKLOAD_GROUPS:
        return BENCHMARK_DENSE_MODES

    return STANDARD_DENSE_MODES


def sample_expanded_workloads_for_group(group_name: str):
    if group_name == "shared_prefix_chat":
        return ["shared_prefix_chat_v0", "shared_prefix_chat_v1"]

    expanded = expand_workload_group_names([group_name])
    expanded = list(dict.fromkeys(expanded))

    limit = WORKLOAD_SAMPLE_LIMITS.get(group_name)

    if limit is None or len(expanded) <= limit:
        return expanded

    stable_digest = hashlib.md5(group_name.encode("utf-8")).hexdigest()
    stable_offset = int(stable_digest[:8], 16)
    rng = random.Random(WORKLOAD_SAMPLE_SEED + stable_offset)
    sampled = rng.sample(expanded, k=limit)

    return sorted(sampled)

In [ ]:
# ============================================================
# Cell 5: Expand sampled final workload plan
# ============================================================

expanded_workloads = []

for group_name in FINAL_DENSE_WORKLOAD_GROUPS:
    expanded_workloads.extend(sample_expanded_workloads_for_group(group_name))

FINAL_DENSE_WORKLOADS = list(dict.fromkeys(expanded_workloads))

TOTAL_DENSE_RUNS = sum(
    len(modes_for_workload(workload_name)) * trials_for_workload(workload_name)
    for workload_name in FINAL_DENSE_WORKLOADS
)

plan_rows = []

for workload_name in FINAL_DENSE_WORKLOADS:
    group_name = canonical_workload_group_name(workload_name)

    for mode_name in modes_for_workload(workload_name):
        plan_rows.append({
            "mode_name": mode_name,
            "workload_name": workload_name,
            "workload_group": group_name,
            "trials": trials_for_workload(workload_name),
        })

plan_df = pd.DataFrame(plan_rows)

group_plan_df = (
    plan_df
    .groupby("workload_group", as_index=False)
    .agg(
        num_expanded_workloads=("workload_name", "nunique"),
        num_modes=("mode_name", "nunique"),
        trials_per_pair=("trials", "max"),
        planned_runs=("trials", "sum"),
    )
)

print("Standard dense modes:")
for mode_name in STANDARD_DENSE_MODES:
    print(" -", mode_name)

print("\nBenchmark-safe dense modes:")
for mode_name in BENCHMARK_DENSE_MODES:
    print(" -", mode_name)

print("\nTotal expanded workloads:", len(FINAL_DENSE_WORKLOADS))
print("Total planned runs:", TOTAL_DENSE_RUNS)

display(group_plan_df)
display(plan_df.head(30))
display(plan_df.tail(30))

In [ ]:
# ============================================================
# Cell 6: Resolve and inspect final mode set
# ============================================================

ALL_FINAL_DENSE_MODES = list(dict.fromkeys(STANDARD_DENSE_MODES + BENCHMARK_DENSE_MODES))

resolved_dense_rows = []

for mode_name in ALL_FINAL_DENSE_MODES:
    mode = resolve_runtime_mode(mode_name)
    resolved_dense_rows.append({
        "mode_name": mode.name,
        "backend": mode.backend,
        "quantization": mode.quantization,
        "prefix_caching": mode.prefix_caching,
        "continuous_batching": mode.continuous_batching,
        "speculative_decoding": mode.speculative_decoding,
        "primary_phase": mode.primary_phase,
        "notes": mode.notes,
    })

resolved_dense_df = pd.DataFrame(resolved_dense_rows)
display(resolved_dense_df)

In [ ]:
# ============================================================
# Cell 7: Execute final sweep (reuse loaded bundle per mode)
# ============================================================

from model_loader import load_model_for_mode, unload_model

dense_results = []
dense_error_rows = []

dense_run_stamp = time.strftime("%Y%m%d_%H%M%S")
dense_partial_json_path = RAW_RESULTS_DIR / f"dense_controller_partial_{dense_run_stamp}.json"
dense_partial_csv_path = RAW_RESULTS_DIR / f"dense_controller_partial_{dense_run_stamp}.csv"

run_counter = 0

# Build per-mode workload list so each mode is loaded once.
mode_to_workloads = {}
for mode_name in ALL_FINAL_DENSE_MODES:
    mode_to_workloads[mode_name] = [
        workload_name
        for workload_name in FINAL_DENSE_WORKLOADS
        if mode_name in modes_for_workload(workload_name)
    ]

try:
    for mode_name in ALL_FINAL_DENSE_MODES:
        runtime_mode = resolve_runtime_mode(mode_name)
        relevant_workloads = mode_to_workloads[mode_name]

        print("\n" + "#" * 120)
        print(f"LOADING MODE ONCE: {mode_name}")
        print(f"Relevant workloads: {len(relevant_workloads)}")
        print("#" * 120)

        bundle = None

        try:
            bundle = load_model_for_mode(runtime_mode)

            for workload_name in relevant_workloads:
                n_trials = trials_for_workload(workload_name)
                runtime_workload = resolve_runtime_workload(workload_name)

                print("\n" + "=" * 120)
                print(f"MODE: {mode_name}")
                print(f"WORKLOAD: {workload_name}")
                print(f"WORKLOAD GROUP: {canonical_workload_group_name(workload_name)}")
                print(f"TRIALS: {n_trials}")
                print("=" * 120)

                for trial_index in range(n_trials):
                    run_counter += 1

                    try:
                        result = run_single_benchmark(
                            runtime_mode=runtime_mode,
                            workload=runtime_workload,
                            trial_index=trial_index,
                            preloaded_bundle=bundle,
                        )

                        dense_results.append(result)

                        print(
                            f"[{run_counter}/{TOTAL_DENSE_RUNS}] "
                            f"{mode_name:28s} | {workload_name:48s} | trial={trial_index:02d} | "
                            f"success={result.success} | "
                            f"ttft={fmt_num(result.ttft_ms)} ms | "
                            f"lat={fmt_num(result.total_latency_ms)} ms | "
                            f"tps={fmt_num(result.tokens_per_second)} | "
                            f"J/tok={fmt_num(result.energy_per_token_j, 3)} | "
                            f"gpu={fmt_num(result.peak_gpu_memory_mb)} MB"
                        )

                    except KeyboardInterrupt:
                        print("\nInterrupted by user. Saving partial outputs before raising.")
                        if dense_results:
                            save_results_json(dense_results, dense_partial_json_path)
                            save_results_csv(dense_results, dense_partial_csv_path)
                            print("Partial JSON:", dense_partial_json_path)
                            print("Partial CSV:", dense_partial_csv_path)
                        raise

                    except Exception as exc:
                        dense_error_rows.append({
                            "mode_name": mode_name,
                            "workload_name": workload_name,
                            "trial_index": trial_index,
                            "error": str(exc),
                        })

                        print(
                            f"[{run_counter}/{TOTAL_DENSE_RUNS}] "
                            f"{mode_name:28s} | {workload_name:48s} | trial={trial_index:02d} | "
                            f"ERROR={exc}"
                        )

                    if run_counter % 25 == 0 and dense_results:
                        save_results_json(dense_results, dense_partial_json_path)
                        save_results_csv(dense_results, dense_partial_csv_path)
                        print(f"Checkpoint saved -> {dense_partial_json_path}")
                        print(f"Checkpoint saved -> {dense_partial_csv_path}")

                    if run_counter % 10 == 0:
                        gc.collect()
                        if torch is not None and torch.cuda.is_available():
                            torch.cuda.empty_cache()

        finally:
            if bundle is not None:
                unload_model(bundle)
                gc.collect()
                if torch is not None and torch.cuda.is_available():
                    torch.cuda.empty_cache()

finally:
    print(f"\nCollected {len(dense_results)} result objects so far.")
    print(f"Logged {len(dense_error_rows)} hard errors outside result objects.")

===================================================================================

In [ ]:
# ============================================================
# Cell 13: Generate final reporter bundle
# ============================================================

# Find the newest completed/prejudge/partial dense JSON automatically.
candidate_jsons = (
    sorted(RAW_RESULTS_DIR.glob("dense_controller_results_*.json"))
    + sorted(RAW_RESULTS_DIR.glob("dense_controller_prejudge_results_*.json"))
    + sorted(RAW_RESULTS_DIR.glob("dense_controller_partial_*.json"))
    + sorted(RAW_RESULTS_DIR.glob("dense_final_results_*.json"))
    + sorted(RAW_RESULTS_DIR.glob("dense_final_prejudge_results_*.json"))
    + sorted(RAW_RESULTS_DIR.glob("dense_final_partial_*.json"))
)

if not candidate_jsons:
    raise FileNotFoundError(f"No dense controller or dense final JSON files found in {RAW_RESULTS_DIR}")

dense_json_path = candidate_jsons[-1]

dense_final_stamp = (
    dense_json_path.stem
    .replace("dense_controller_results_", "")
    .replace("dense_controller_prejudge_results_", "")
    .replace("dense_controller_partial_", "")
    .replace("dense_final_results_", "")
    .replace("dense_final_prejudge_results_", "")
    .replace("dense_final_partial_", "")
)

print("Using dense JSON:", dense_json_path)
print("Dense final stamp:", dense_final_stamp)

dense_report_dir = generate_full_report(
    input_path=dense_json_path,
    output_dir=RAW_RESULTS_DIR / f"dense_controller_report_{dense_final_stamp}",
    quality_metric="auto",
)

dense_report_dir = Path(dense_report_dir)

print("Dense report directory:", dense_report_dir)

dense_agg_df = pd.read_csv(dense_report_dir / "aggregated.csv")
dense_delta_df = pd.read_csv(dense_report_dir / "deltas.csv")
dense_phase_df = pd.read_csv(dense_report_dir / "phase_dominance.csv")
dense_failure_df = pd.read_csv(dense_report_dir / "failure_summary.csv")

print("Aggregated rows:", len(dense_agg_df))
print("Delta rows:", len(dense_delta_df))
print("Phase rows:", len(dense_phase_df))
print("Failure rows:", len(dense_failure_df))

In [ ]:
# ============================================================
# Cell 14: Tables + plotting setup
# ============================================================

PLOT_DIR = RAW_RESULTS_DIR.parent / "plots" / f"dense_controller_plots_{dense_final_stamp}"
TABLE_DIR = RAW_RESULTS_DIR.parent / "processed" / f"dense_controller_tables_{dense_final_stamp}"

PLOT_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

MODE_ORDER = [
    "fp16_baseline",
    "gptq_4bit",
    "speculative_decoding",
    "gptq_plus_prefix_caching",
    "int8_plus_continuous_batching",
    "prefix_caching",
    "continuous_batching",
    "controller_v1",
]
MODE_ORDER = [m for m in MODE_ORDER if m in set(dense_agg_df["mode_name"])]

BENCHMARK_MODE_ORDER = [
    "fp16_baseline",
    "gptq_4bit",
    "int8_quant",
    "speculative_decoding",
    "prefix_caching",
    "controller_v1",
]

BENCHMARK_MODE_ORDER = [m for m in BENCHMARK_MODE_ORDER if m in set(dense_agg_df["mode_name"])]

WORKLOAD_ORDER_REPORT = [
    "short_prompt_short_output",
    "short_prompt_long_output",
    "long_prompt_short_output",
    "long_prompt_long_output",
    "shared_prefix_chat_v0",
    "shared_prefix_chat_v1",
    "memory_pressure_long_context",
    "mmlu_pro_eval",
    "gsm8k_eval",
    "truthfulqa_eval",
    "gpqa_eval",
    "mlu_eval",
    "mt_bench_eval",
    "alpacaeval2_lc_eval",
]
WORKLOAD_ORDER_REPORT = [w for w in WORKLOAD_ORDER_REPORT if w in set(dense_agg_df["workload_name"])]

AUTOMATIC_BENCHMARK_DISPLAY_ORDER = [
    "mmlu_pro_eval",
    "gsm8k_eval",
    "truthfulqa_eval",
    "gpqa_eval",
    "mlu_eval",
]
AUTOMATIC_BENCHMARK_DISPLAY_ORDER = [
    w for w in AUTOMATIC_BENCHMARK_DISPLAY_ORDER
    if w in set(dense_agg_df["workload_name"])
]

JUDGE_BENCHMARK_DISPLAY_ORDER = [
    "mt_bench_eval",
    "alpacaeval2_lc_eval",
]
JUDGE_BENCHMARK_DISPLAY_ORDER = [
    w for w in JUDGE_BENCHMARK_DISPLAY_ORDER
    if w in set(dense_agg_df["workload_name"])
]

BENCHMARK_LABELS = {
    "mmlu_pro_eval": "MMLU-Pro",
    "gsm8k_eval": "GSM8K",
    "truthfulqa_eval": "TruthfulQA",
    "gpqa_eval": "GPQA",
    "mlu_eval": "MLU/MMMLU",
    "mt_bench_eval": "MT-Bench-style",
    "alpacaeval2_lc_eval": "AlpacaEval-style",
}

def save_table(df, name):
    path = TABLE_DIR / name
    df.to_csv(path, index=False)
    print("Saved:", path)
    return path

def plot_annotated_heatmap(
    df_matrix,
    title,
    cbar_label,
    fmt="{:.1f}",
    figsize=(12, 6),
    cmap="viridis",
    save_name=None,
):
    matrix = df_matrix.values.astype(float)

    if not np.isfinite(matrix).any():
        print(f"Skipping empty heatmap: {title}")
        return None, None

    fig, ax = plt.subplots(figsize=figsize)
    masked = np.ma.masked_invalid(matrix)

    im = ax.imshow(masked, aspect="auto", cmap=cmap)

    ax.set_title(title)
    ax.set_xticks(np.arange(df_matrix.shape[1]))
    ax.set_xticklabels(df_matrix.columns, rotation=30, ha="right")
    ax.set_yticks(np.arange(df_matrix.shape[0]))
    ax.set_yticklabels(df_matrix.index)

    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label(cbar_label)

    for i in range(df_matrix.shape[0]):
        for j in range(df_matrix.shape[1]):
            value = df_matrix.iloc[i, j]
            label = "–" if pd.isna(value) else fmt.format(value)
            ax.text(j, i, label, ha="center", va="center", fontsize=8)

    plt.tight_layout()

    if save_name is not None:
        save_path = PLOT_DIR / save_name
        plt.savefig(save_path, dpi=200, bbox_inches="tight")
        print("Saved:", save_path)

    plt.show()
    return fig, ax

def plot_bar_metric(df, x_col, y_col, title, ylabel, save_name=None, rotation=25, ylim=None):
    fig, ax = plt.subplots(figsize=(9, 4.5))

    x = np.arange(len(df))
    y = df[y_col].values

    ax.bar(x, y)
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.set_xticks(x)
    ax.set_xticklabels(df[x_col].values, rotation=rotation, ha="right")

    if ylim is not None:
        ax.set_ylim(*ylim)

    for i, value in enumerate(y):
        if pd.notna(value):
            ax.text(i, value, f"{value:.2f}", ha="center", va="bottom", fontsize=8)

    plt.tight_layout()

    if save_name is not None:
        save_path = PLOT_DIR / save_name
        plt.savefig(save_path, dpi=200, bbox_inches="tight")
        print("Saved:", save_path)

    plt.show()
    return fig, ax

print("Plot dir:", PLOT_DIR)
print("Table dir:", TABLE_DIR)

In [ ]:
# ============================================================
# Cell 15: Performance table with baseline ratios
# ============================================================

needed_cols = [
    "mode_name",
    "workload_name",
    "system_condition",
    "n",
    "failure_count",
    "ttft_ms_mean",
    "avg_tbt_ms_mean",
    "total_latency_ms_mean",
    "tokens_per_second_mean",
    "peak_gpu_memory_mb_mean",
    "energy_per_token_j_mean",
    "benchmark_primary_metric_value_mean",
    "reference_rouge_l_f1_mean",
]
needed_cols = [c for c in needed_cols if c in dense_agg_df.columns]

dense_perf_df = dense_agg_df[needed_cols].copy()

baseline_df = (
    dense_perf_df[dense_perf_df["mode_name"] == "fp16_baseline"]
    [
        [
            "workload_name",
            "total_latency_ms_mean",
            "tokens_per_second_mean",
            "energy_per_token_j_mean",
            "ttft_ms_mean",
            "avg_tbt_ms_mean",
        ]
    ]
    .rename(
        columns={
            "total_latency_ms_mean": "baseline_total_latency_ms_mean",
            "tokens_per_second_mean": "baseline_tokens_per_second_mean",
            "energy_per_token_j_mean": "baseline_energy_per_token_j_mean",
            "ttft_ms_mean": "baseline_ttft_ms_mean",
            "avg_tbt_ms_mean": "baseline_avg_tbt_ms_mean",
        }
    )
)

dense_perf_df = dense_perf_df.merge(baseline_df, on="workload_name", how="left")

dense_perf_df["latency_speedup_vs_baseline"] = (
    dense_perf_df["baseline_total_latency_ms_mean"] / dense_perf_df["total_latency_ms_mean"]
)
dense_perf_df["throughput_ratio_vs_baseline"] = (
    dense_perf_df["tokens_per_second_mean"] / dense_perf_df["baseline_tokens_per_second_mean"]
)
dense_perf_df["energy_ratio_vs_baseline"] = (
    dense_perf_df["energy_per_token_j_mean"] / dense_perf_df["baseline_energy_per_token_j_mean"]
)
dense_perf_df["energy_reduction_pct_vs_baseline"] = (
    100.0 * (1.0 - dense_perf_df["energy_ratio_vs_baseline"])
)

dense_perf_df = dense_perf_df.sort_values(["workload_name", "total_latency_ms_mean"])

save_table(dense_perf_df, "dense_performance_with_baseline_ratios.csv")

display(dense_perf_df.head(30))

In [ ]:
# ============================================================
# Cell 16: Accuracy and judge-quality tables
# ============================================================

AUTOMATIC_BENCHMARK_METRIC_MAP = {
    "mmlu_pro_eval": "mmlu_pro_accuracy_mean",
    "gsm8k_eval": "gsm8k_exact_match_accuracy_mean",
    "truthfulqa_eval": "truthfulqa_accuracy_mean",
    "gpqa_eval": "gpqa_accuracy_mean",
    "mlu_eval": "mlu_accuracy_mean",
}

JUDGE_BENCHMARK_METRIC_MAP = {
    "mt_bench_eval": "mt_bench_score_mean",
    "alpacaeval2_lc_eval": "alpacaeval2_lc_win_rate_mean",
}

def build_benchmark_metric_table(metric_map, display_as_percent=True):
    rows = []

    for workload_name, metric_col in metric_map.items():
        if workload_name not in set(dense_agg_df["workload_name"]):
            print(f"Skipping {workload_name}: workload missing")
            continue

        if metric_col not in dense_agg_df.columns:
            print(f"Skipping {workload_name}: metric missing -> {metric_col}")
            continue

        sub = dense_agg_df[dense_agg_df["workload_name"] == workload_name].copy()
        sub["benchmark"] = workload_name
        sub["benchmark_label"] = sub["benchmark"].map(BENCHMARK_LABELS).fillna(sub["benchmark"])
        sub["metric_column"] = metric_col
        sub["metric_value"] = sub[metric_col]

        if display_as_percent:
            sub["metric_display_value"] = 100.0 * sub["metric_value"]
            sub["metric_display_unit"] = "%"
        else:
            sub["metric_display_value"] = sub["metric_value"]
            sub["metric_display_unit"] = "score"

        rows.append(
            sub[
                [
                    "benchmark",
                    "benchmark_label",
                    "mode_name",
                    "n",
                    "failure_count",
                    "total_latency_ms_mean",
                    "tokens_per_second_mean",
                    "energy_per_token_j_mean",
                    "benchmark_primary_metric_value_mean",
                    "metric_column",
                    "metric_value",
                    "metric_display_value",
                    "metric_display_unit",
                ]
            ]
        )

    if not rows:
        return pd.DataFrame()

    out = pd.concat(rows, ignore_index=True)
    out = out.sort_values(
        ["benchmark", "metric_display_value", "total_latency_ms_mean"],
        ascending=[True, False, True],
    )
    return out

automatic_accuracy_df = build_benchmark_metric_table(
    AUTOMATIC_BENCHMARK_METRIC_MAP,
    display_as_percent=True,
)

judge_quality_df = build_benchmark_metric_table(
    JUDGE_BENCHMARK_METRIC_MAP,
    display_as_percent=False,
)

save_table(automatic_accuracy_df, "automatic_accuracy_table.csv")
save_table(judge_quality_df, "judge_quality_table.csv")

print("Automatic accuracy:")
display(automatic_accuracy_df)

print("Judge quality:")
display(judge_quality_df)

In [ ]:
# ============================================================
# Cell 17: Separate automatic accuracy and judge-quality plots
# ============================================================

if len(automatic_accuracy_df) > 0:
    automatic_accuracy_matrix = (
        automatic_accuracy_df
        .pivot(index="mode_name", columns="benchmark_label", values="metric_display_value")
        .reindex(
            index=BENCHMARK_MODE_ORDER,
            columns=[
                BENCHMARK_LABELS[w]
                for w in AUTOMATIC_BENCHMARK_DISPLAY_ORDER
                if w in BENCHMARK_LABELS
            ],
        )
    )

    plot_annotated_heatmap(
        automatic_accuracy_matrix,
        title="Automatic Benchmark Accuracy by Mode",
        cbar_label="Accuracy / Exact Match (%)",
        fmt="{:.1f}",
        figsize=(10, 4.5),
        cmap="viridis",
        save_name="automatic_accuracy_heatmap.png",
    )

if len(judge_quality_df) > 0:
    for benchmark_name in JUDGE_BENCHMARK_DISPLAY_ORDER:
        benchmark_label = BENCHMARK_LABELS.get(benchmark_name, benchmark_name)

        sub = (
            judge_quality_df[judge_quality_df["benchmark"] == benchmark_name]
            .set_index("mode_name")
            .reindex(BENCHMARK_MODE_ORDER)
            .reset_index()
        )

        if sub["metric_display_value"].notna().sum() == 0:
            print("Skipping empty judge plot:", benchmark_label)
            continue

        if benchmark_name == "mt_bench_eval":
            ylabel = "Mean judge score"
            ylim = (0, 10)
        else:
            ylabel = "Win rate vs FP16 baseline"
            ylim = (0, 1)

        plot_bar_metric(
            sub,
            x_col="mode_name",
            y_col="metric_display_value",
            title=f"{benchmark_label} Judge Quality by Mode",
            ylabel=ylabel,
            ylim=ylim,
            save_name=f"judge_{benchmark_name}_quality_bar.png",
        )

In [ ]:
# ============================================================
# Cell 18: Performance heatmaps
# ============================================================

for value_col, title, cbar_label, fmt, save_name in [
    ("total_latency_ms_mean", "Mean Total Latency by Mode and Workload", "Total latency (ms)", "{:.0f}", "total_latency_heatmap.png"),
    ("tokens_per_second_mean", "Mean Throughput by Mode and Workload", "Tokens per second", "{:.1f}", "throughput_heatmap.png"),
    ("energy_per_token_j_mean", "Mean Energy per Token by Mode and Workload", "Energy per token (J/token)", "{:.2f}", "energy_per_token_heatmap.png"),
    ("peak_gpu_memory_mb_mean", "Peak GPU Memory by Mode and Workload", "Peak GPU memory (MB)", "{:.0f}", "peak_gpu_memory_heatmap.png"),
]:
    matrix = (
        dense_agg_df
        .pivot(index="mode_name", columns="workload_name", values=value_col)
        .reindex(index=MODE_ORDER, columns=WORKLOAD_ORDER_REPORT)
    )

    plot_annotated_heatmap(
        matrix,
        title=title,
        cbar_label=cbar_label,
        fmt=fmt,
        figsize=(14, 6),
        cmap="viridis",
        save_name=save_name,
    )

In [ ]:
# ============================================================
# Cell 19: Baseline ratio heatmaps
# ============================================================

speedup_matrix = (
    dense_perf_df
    .pivot(index="mode_name", columns="workload_name", values="latency_speedup_vs_baseline")
    .reindex(index=MODE_ORDER, columns=WORKLOAD_ORDER_REPORT)
)

plot_annotated_heatmap(
    speedup_matrix,
    title="Latency Speedup vs FP16 Baseline",
    cbar_label="Speedup factor, higher is better",
    fmt="{:.2f}×",
    figsize=(14, 6),
    cmap="viridis",
    save_name="latency_speedup_vs_baseline_heatmap.png",
)

energy_ratio_matrix = (
    dense_perf_df
    .pivot(index="mode_name", columns="workload_name", values="energy_ratio_vs_baseline")
    .reindex(index=MODE_ORDER, columns=WORKLOAD_ORDER_REPORT)
)

plot_annotated_heatmap(
    energy_ratio_matrix,
    title="Energy Ratio vs FP16 Baseline",
    cbar_label="Energy ratio, lower is better",
    fmt="{:.2f}×",
    figsize=(14, 6),
    cmap="viridis",
    save_name="energy_ratio_vs_baseline_heatmap.png",
)

In [ ]:
# ============================================================
# Cell 20: Phase dominance heatmaps
# ============================================================

prefill_matrix = (
    dense_phase_df
    .pivot(index="mode_name", columns="workload_name", values="prefill_pct")
    .reindex(index=MODE_ORDER, columns=WORKLOAD_ORDER_REPORT)
)

decode_matrix = (
    dense_phase_df
    .pivot(index="mode_name", columns="workload_name", values="decode_pct")
    .reindex(index=MODE_ORDER, columns=WORKLOAD_ORDER_REPORT)
)

plot_annotated_heatmap(
    prefill_matrix,
    title="Prefill Share of Total Latency",
    cbar_label="Prefill share (%)",
    fmt="{:.1f}",
    figsize=(14, 6),
    cmap="Blues",
    save_name="prefill_share_heatmap.png",
)

plot_annotated_heatmap(
    decode_matrix,
    title="Decode Share of Total Latency",
    cbar_label="Decode share (%)",
    fmt="{:.1f}",
    figsize=(14, 6),
    cmap="Oranges",
    save_name="decode_share_heatmap.png",
)

In [ ]:
# ============================================================
# Cell 21: Best mode tables
# ============================================================

best_latency_df = (
    dense_perf_df
    .sort_values(["workload_name", "total_latency_ms_mean"])
    .groupby("workload_name", as_index=False)
    .first()
)

best_energy_df = (
    dense_perf_df
    .sort_values(["workload_name", "energy_per_token_j_mean"])
    .groupby("workload_name", as_index=False)
    .first()
)

best_throughput_df = (
    dense_perf_df
    .sort_values(["workload_name", "tokens_per_second_mean"], ascending=[True, False])
    .groupby("workload_name", as_index=False)
    .first()
)

best_latency_df = best_latency_df[
    [
        "workload_name",
        "mode_name",
        "total_latency_ms_mean",
        "latency_speedup_vs_baseline",
        "tokens_per_second_mean",
        "energy_per_token_j_mean",
        "energy_ratio_vs_baseline",
        "peak_gpu_memory_mb_mean",
    ]
]

best_energy_df = best_energy_df[
    [
        "workload_name",
        "mode_name",
        "energy_per_token_j_mean",
        "energy_ratio_vs_baseline",
        "total_latency_ms_mean",
        "latency_speedup_vs_baseline",
        "tokens_per_second_mean",
        "peak_gpu_memory_mb_mean",
    ]
]

best_throughput_df = best_throughput_df[
    [
        "workload_name",
        "mode_name",
        "tokens_per_second_mean",
        "throughput_ratio_vs_baseline",
        "total_latency_ms_mean",
        "latency_speedup_vs_baseline",
        "energy_per_token_j_mean",
        "peak_gpu_memory_mb_mean",
    ]
]

save_table(best_latency_df, "best_mode_by_latency.csv")
save_table(best_energy_df, "best_mode_by_energy.csv")
save_table(best_throughput_df, "best_mode_by_throughput.csv")

print("Best by latency:")
display(best_latency_df)

print("Best by energy:")
display(best_energy_df)

print("Best by throughput:")
display(best_throughput_df)

In [ ]:
# ============================================================
# Cell 22: List saved plots and tables
# ============================================================

print("Saved plots:")
for path in sorted(PLOT_DIR.glob("*")):
    print(" -", path)

print("\nSaved tables:")
for path in sorted(TABLE_DIR.glob("*")):
    print(" -", path)

print("\nFinal dense result files:")
maybe_paths = [
    globals().get("dense_json_path"),
    globals().get("dense_csv_path"),
    globals().get("dense_summary_csv_path"),
    globals().get("dense_aggregate_csv_path"),
    globals().get("dense_comparison_csv_path"),
    globals().get("dense_error_csv_path"),
    globals().get("dense_report_dir"),
    globals().get("judge_sidecar_path"),
]

for path in maybe_paths:
    if path is not None:
        print(" -", path)

In [ ]:
1

## Phase-aware fixed-mode routing controller

The project changed after the mid-request switching plan broke the KV cache. The controller now routes at the request boundary instead. It reads static signals, picks one fixed mode up front, and keeps the request on that mode for both phases. The quality target here is simple: the controller passes if its mean accuracy across the 5 auto-graded evals stays within 1.5 points of the FP16 baseline.

This sanity check confirms that the sweep actually produced aggregated rows for `controller_v1`. If the run was complete, it should cover all 14 workloads.

In [ ]:
controller_rows_df = dense_agg_df[dense_agg_df["mode_name"] == "controller_v1"].copy()
controller_workloads = sorted(controller_rows_df["workload_name"].dropna().unique().tolist())

print("Controller row count:", len(controller_rows_df))
print("Controller workloads covered:")
for workload_name in controller_workloads:
    print(" -", workload_name)

controller_rows_df[["workload_name", "system_condition"]].sort_values(["workload_name", "system_condition"]).reset_index(drop=True)

This table shows which underlying mode the controller actually picked for each workload. It also shows the controller phase label, the estimated prefill share, and the fixed-rule explanation.

In [ ]:
controller_prefill_col = (
    "controller_estimated_prefill_share_pct"
    if "controller_estimated_prefill_share_pct" in dense_agg_df.columns
    else "controller_estimated_prefill_share_pct_mean"
)

if {
    "controller_selected_mode_name",
    "controller_phase_label",
    "controller_route_reason",
}.issubset(dense_agg_df.columns):
    controller_routing_df = (
        dense_agg_df[dense_agg_df["mode_name"] == "controller_v1"]
        [[
            "workload_name",
            "controller_selected_mode_name",
            "controller_phase_label",
            controller_prefill_col,
            "controller_route_reason",
        ]]
        .rename(columns={controller_prefill_col: "controller_estimated_prefill_share_pct"})
        .sort_values("workload_name")
        .reset_index(drop=True)
    )
else:
    with open(dense_json_path, "r", encoding="utf-8") as f:
        dense_raw_rows = json.load(f)

    dense_raw_df = pd.DataFrame(dense_raw_rows)
    dense_raw_df["workload_name_group"] = dense_raw_df["workload_name"].astype(str).str.split("__", n=1).str[0]

    controller_routing_df = (
        dense_raw_df[dense_raw_df["mode_name"] == "controller_v1"]
        [[
            "workload_name_group",
            "controller_selected_mode_name",
            "controller_phase_label",
            "controller_estimated_prefill_share_pct",
            "controller_route_reason",
        ]]
        .rename(columns={"workload_name_group": "workload_name"})
        .sort_values("workload_name")
        .drop_duplicates(subset=["workload_name"], keep="first")
        .reset_index(drop=True)
    )

controller_routing_df

This compares `controller_v1` against `fp16_baseline` on the core performance metrics. Latency speedup is FP16 divided by controller latency, so higher is better. The other ratios are controller divided by FP16.

In [ ]:
controller_fp16_compare_df = (
    dense_agg_df[dense_agg_df["mode_name"] == "controller_v1"]
    [[
        "workload_name",
        "total_latency_ms_mean",
        "tokens_per_second_mean",
        "energy_per_token_j_mean",
        "peak_gpu_memory_mb_mean",
    ]]
    .rename(columns={
        "total_latency_ms_mean": "controller_total_latency_ms_mean",
        "tokens_per_second_mean": "controller_tokens_per_second_mean",
        "energy_per_token_j_mean": "controller_energy_per_token_j_mean",
        "peak_gpu_memory_mb_mean": "controller_peak_gpu_memory_mb_mean",
    })
    .merge(
        dense_agg_df[dense_agg_df["mode_name"] == "fp16_baseline"]
        [[
            "workload_name",
            "total_latency_ms_mean",
            "tokens_per_second_mean",
            "energy_per_token_j_mean",
            "peak_gpu_memory_mb_mean",
        ]]
        .rename(columns={
            "total_latency_ms_mean": "fp16_total_latency_ms_mean",
            "tokens_per_second_mean": "fp16_tokens_per_second_mean",
            "energy_per_token_j_mean": "fp16_energy_per_token_j_mean",
            "peak_gpu_memory_mb_mean": "fp16_peak_gpu_memory_mb_mean",
        }),
        on="workload_name",
        how="inner",
    )
)

controller_fp16_compare_df["latency_speedup"] = (
    controller_fp16_compare_df["fp16_total_latency_ms_mean"]
    / controller_fp16_compare_df["controller_total_latency_ms_mean"]
)
controller_fp16_compare_df["throughput_ratio"] = (
    controller_fp16_compare_df["controller_tokens_per_second_mean"]
    / controller_fp16_compare_df["fp16_tokens_per_second_mean"]
)
controller_fp16_compare_df["energy_ratio"] = (
    controller_fp16_compare_df["controller_energy_per_token_j_mean"]
    / controller_fp16_compare_df["fp16_energy_per_token_j_mean"]
)
controller_fp16_compare_df["memory_ratio"] = (
    controller_fp16_compare_df["controller_peak_gpu_memory_mb_mean"]
    / controller_fp16_compare_df["fp16_peak_gpu_memory_mb_mean"]
)

controller_fp16_compare_df = controller_fp16_compare_df.sort_values("latency_speedup", ascending=False).reset_index(drop=True)
controller_fp16_compare_df

This checks the quality gate on the 5 auto-graded evals. Each accuracy is shown in percentage points, then the mean controller minus FP16 delta is tested against the 1.5 point threshold.

In [ ]:
controller_accuracy_metric_map = {
    "mmlu_pro_eval": "mmlu_pro_accuracy_mean",
    "gsm8k_eval": "gsm8k_exact_match_accuracy_mean",
    "truthfulqa_eval": "truthfulqa_accuracy_mean",
    "gpqa_eval": "gpqa_accuracy_mean",
    "mlu_eval": "mlu_accuracy_mean",
}

controller_auto_accuracy_rows = []

for workload_name, metric_col in controller_accuracy_metric_map.items():
    sub = dense_agg_df[dense_agg_df["workload_name"] == workload_name].copy()
    controller_row = sub[sub["mode_name"] == "controller_v1"]
    fp16_row = sub[sub["mode_name"] == "fp16_baseline"]

    if len(controller_row) == 0 or len(fp16_row) == 0:
        continue

    controller_accuracy_pct = 100.0 * float(controller_row.iloc[0][metric_col])
    fp16_accuracy_pct = 100.0 * float(fp16_row.iloc[0][metric_col])

    controller_auto_accuracy_rows.append({
        "workload_name": workload_name,
        "metric_column": metric_col,
        "controller_accuracy_pct": controller_accuracy_pct,
        "fp16_accuracy_pct": fp16_accuracy_pct,
        "delta_pts": controller_accuracy_pct - fp16_accuracy_pct,
    })

controller_auto_accuracy_df = pd.DataFrame(controller_auto_accuracy_rows).sort_values("workload_name").reset_index(drop=True)
mean_controller_auto_accuracy_pct = controller_auto_accuracy_df["controller_accuracy_pct"].mean()
mean_fp16_auto_accuracy_pct = controller_auto_accuracy_df["fp16_accuracy_pct"].mean()
mean_accuracy_delta_pts = mean_controller_auto_accuracy_pct - mean_fp16_auto_accuracy_pct
controller_quality_pass = abs(mean_accuracy_delta_pts) <= 1.5

print("Mean controller auto-graded accuracy:", round(mean_controller_auto_accuracy_pct, 3))
print("Mean FP16 auto-graded accuracy:", round(mean_fp16_auto_accuracy_pct, 3))
print("Mean accuracy delta (controller - FP16):", round(mean_accuracy_delta_pts, 3))
print("Quality threshold result:", "PASS" if controller_quality_pass else "FAIL")

controller_auto_accuracy_df

This compares the controller against an oracle that always picks the best non-controller mode after seeing the results. The ratio columns show how much performance the fixed rule leaves on the table relative to perfect hindsight routing.

In [ ]:
non_controller_df = dense_agg_df[dense_agg_df["mode_name"] != "controller_v1"].copy()

latency_oracle_df = (
    non_controller_df
    .sort_values(["workload_name", "total_latency_ms_mean"])
    .groupby("workload_name", as_index=False)
    .first()
    [["workload_name", "mode_name", "total_latency_ms_mean"]]
    .rename(columns={
        "mode_name": "latency_oracle_mode_name",
        "total_latency_ms_mean": "latency_oracle_total_latency_ms_mean",
    })
)

energy_oracle_df = (
    non_controller_df
    .sort_values(["workload_name", "energy_per_token_j_mean"])
    .groupby("workload_name", as_index=False)
    .first()
    [["workload_name", "mode_name", "energy_per_token_j_mean"]]
    .rename(columns={
        "mode_name": "energy_oracle_mode_name",
        "energy_per_token_j_mean": "energy_oracle_energy_per_token_j_mean",
    })
)

controller_oracle_compare_df = (
    controller_fp16_compare_df
    [[
        "workload_name",
        "controller_total_latency_ms_mean",
        "controller_energy_per_token_j_mean",
    ]]
    .merge(controller_routing_df[["workload_name", "controller_selected_mode_name"]], on="workload_name", how="left")
    .merge(latency_oracle_df, on="workload_name", how="left")
    .merge(energy_oracle_df, on="workload_name", how="left")
)

controller_oracle_compare_df["latency_oracle_over_controller"] = (
    controller_oracle_compare_df["latency_oracle_total_latency_ms_mean"]
    / controller_oracle_compare_df["controller_total_latency_ms_mean"]
)
controller_oracle_compare_df["energy_oracle_over_controller"] = (
    controller_oracle_compare_df["energy_oracle_energy_per_token_j_mean"]
    / controller_oracle_compare_df["controller_energy_per_token_j_mean"]
)
controller_oracle_compare_df["matches_latency_oracle_mode"] = (
    controller_oracle_compare_df["controller_selected_mode_name"]
    == controller_oracle_compare_df["latency_oracle_mode_name"]
)
controller_oracle_compare_df["matches_energy_oracle_mode"] = (
    controller_oracle_compare_df["controller_selected_mode_name"]
    == controller_oracle_compare_df["energy_oracle_mode_name"]
)

controller_oracle_compare_df = controller_oracle_compare_df.sort_values("workload_name").reset_index(drop=True)
controller_oracle_compare_df

These charts show the controller against FP16 on latency speedup and energy ratio. The dashed line at 1.0 marks parity with the baseline.

In [ ]:
controller_plot_df = controller_fp16_compare_df.sort_values("latency_speedup", ascending=False).reset_index(drop=True)

fig, axes = plt.subplots(1, 2, figsize=(18, 5), sharex=True)

axes[0].bar(controller_plot_df["workload_name"], controller_plot_df["latency_speedup"], color="#4c78a8")
axes[0].axhline(1.0, color="black", linestyle="--", linewidth=1)
axes[0].set_title("Controller latency speedup vs FP16")
axes[0].set_ylabel("FP16 / controller latency")
axes[0].tick_params(axis="x", rotation=70)

axes[1].bar(controller_plot_df["workload_name"], controller_plot_df["energy_ratio"], color="#f58518")
axes[1].axhline(1.0, color="black", linestyle="--", linewidth=1)
axes[1].set_title("Controller energy ratio vs FP16")
axes[1].set_ylabel("Controller / FP16 energy")
axes[1].tick_params(axis="x", rotation=70)

plt.tight_layout()
plt.show()

This is the short summary for the controller. It prints the mean speedups, mean energy ratio, mean accuracy delta, and the PASS or FAIL result.

In [ ]:
headline_summary_df = pd.DataFrame([
    {
        "mean_latency_speedup": controller_fp16_compare_df["latency_speedup"].mean(),
        "mean_throughput_ratio": controller_fp16_compare_df["throughput_ratio"].mean(),
        "mean_energy_ratio": controller_fp16_compare_df["energy_ratio"].mean(),
        "mean_accuracy_delta_pts": mean_accuracy_delta_pts,
        "quality_threshold_pass": "PASS" if controller_quality_pass else "FAIL",
    }
])

print("Mean latency speedup:", round(float(headline_summary_df.iloc[0]["mean_latency_speedup"]), 4))
print("Mean throughput ratio:", round(float(headline_summary_df.iloc[0]["mean_throughput_ratio"]), 4))
print("Mean energy ratio:", round(float(headline_summary_df.iloc[0]["mean_energy_ratio"]), 4))
print("Mean accuracy delta (controller - FP16):", round(float(headline_summary_df.iloc[0]["mean_accuracy_delta_pts"]), 4))
print("Quality threshold result:", headline_summary_df.iloc[0]["quality_threshold_pass"])

headline_summary_df

The controller keeps the routing rule simple and interpretable. If the quality threshold holds on the full run, the result is that request-boundary routing preserves FP16-like quality while still improving latency and energy. The oracle table then shows how far the fixed rule still is from perfect hindsight routing.